In [1]:
from dask.distributed import LocalCluster, Client
cluster = LocalCluster()
client = Client(cluster)

In [2]:
import os
foldername = '/home/edavenport/analysis/vel-assim-manuscript/forecasts/results_figs/time_series/'
os.makedirs(foldername, exist_ok=True)

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
import pandas as pd
import numpy as np
from forecasts.forecast_utils import get_forecast_params

# ═══════════════════════════════════════════════════════════════════
#  FORECAST PARAMETERS — only change start_date to switch forecasts
#  Supported range: Sep 2012 – Jul 2013 (1st of month)
# ═══════════════════════════════════════════════════════════════════
start_date = pd.Timestamp('2013-05-01')

# TPOSE-Vel state estimate directory (varies by run; set manually for each forecast)
vel_estimate_data_dir = '/data/SO3/edavenport/tpose6/jan2013/run_iter14'

p = get_forecast_params(start_date)

# ── Unpack for use in subsequent cells ───────────────────────────
start_date  = p.start_date
end_date    = p.end_date
month_str   = p.month_str
day_str     = p.day_str
year_str    = p.year_str

noTAO_data_dir          = p.noTAO_data_dir
noTAO_forecast_data_dir = p.noTAO_forecast_data_dir
vel_forecast_data_dir   = p.vel_forecast_data_dir
grid_dir                = p.grid_dir

ref_date        = p.ref_date
itPerFile       = p.itPerFile
delta_t         = p.delta_t
num_diags       = p.num_diags
intervals       = p.intervals
n_forecast_days = p.n_forecast_days
n_eval          = p.n_eval
eval_slice      = p.eval_slice
eval_start_date = p.eval_start_date
days            = p.days
eval_dates      = p.eval_dates
eval_months     = p.eval_months
month_bounds    = p.month_bounds
month_centers   = p.month_centers

# TAO mooring locations
tao_lons       = np.array([190., 220., 250.])   # °E
tao_lon_labels = ['170°W', '140°W', '110°W']

print(f'Forecast: {start_date.date()} → {end_date.date()} ({n_forecast_days} days)')
print(f'itPerFile: {itPerFile}  |  delta_t: {delta_t:.0f} s')
print(f'Vel est dir: {vel_estimate_data_dir}')

### Load TPOSE

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr
from xmitgcm import open_mdsdataset
plt.rcParams['font.size'] = 11

prefix = ['diag_state']

def open_tpose(data_dir):
    ds = open_mdsdataset(
        data_dir=data_dir, grid_dir=grid_dir,
        iters=intervals, prefix=prefix, ref_date=ref_date, delta_t=delta_t)
    for coord in ['XC', 'YC', 'Z', 'XG', 'YG']:
        ds[coord] = ds[coord].astype(float)
    return ds

ds_tpose_noTAO          = open_tpose(noTAO_data_dir)
ds_tpose_noTAO_forecast = open_tpose(noTAO_forecast_data_dir)
ds_tpose_vel            = open_tpose(vel_estimate_data_dir)
ds_tpose_vel_forecast   = open_tpose(vel_forecast_data_dir)

### Load GLORYS

In [5]:
from forecasts.forecast_utils import load_hycom_daily

glorys = xr.open_mfdataset('/data/SO3/edavenport/tpose6/glorys_data/glorys_*.nc', combine='by_coords')
hycom  = load_hycom_daily(start_date, end_date)   # None if no files exist for this window

glorys_wind = glorys[['uo', 'vo']].sel(
    time=slice(start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d')),
    latitude=slice(-5, 5)
)
if hycom is not None:
    hycom_wind = hycom[['water_u', 'water_v']].sel(lat=slice(-5, 5))

if hycom is None:
    print('WARNING: No HYCOM data available for this forecast window — HYCOM will be omitted from figures.')

### Load TAO ADCP (daily average)

In [6]:
adcp = xr.open_dataset('/data/SO3/edavenport/tpose6/tao_profiles/TAO_WO_2013_ADCP.nc')

raw_dates = pd.to_datetime(
    adcp.prof_YYYYMMDD.values.astype(int).astype(str), format='%Y%m%d'
)
u_raw = adcp.prof_U.values.copy().astype(float)
v_raw = adcp.prof_V.values.copy().astype(float)
u_raw[u_raw == -9999.] = np.nan
v_raw[v_raw == -9999.] = np.nan

adcp_lons   = adcp.prof_lon.values
adcp_depths = adcp.prof_depth.values   # (62,) in metres

# Daily average: (time, location, depth)
tao_u = np.full((n_eval, len(tao_lons), len(adcp_depths)), np.nan)
tao_v = np.full((n_eval, len(tao_lons), len(adcp_depths)), np.nan)

for i, d in enumerate(eval_dates):
    for j, lon in enumerate(tao_lons):
        mask = (raw_dates == d) & (adcp_lons == lon)
        if mask.sum() > 0:
            tao_u[i, j, :] = np.nanmean(u_raw[mask, :], axis=0)
            tao_v[i, j, :] = np.nanmean(v_raw[mask, :], axis=0)

print(f'TAO shape: {tao_u.shape}  (time, location, depth)')

TAO shape: (122, 3, 62)  (time, location, depth)


/tmp/ipykernel_2997962/480107646.py:22: RuntimeWarning: Mean of empty slice
  tao_u[i, j, :] = np.nanmean(u_raw[mask, :], axis=0)
/tmp/ipykernel_2997962/480107646.py:23: RuntimeWarning: Mean of empty slice
  tao_v[i, j, :] = np.nanmean(v_raw[mask, :], axis=0)


### Interpolate TPOSE and GLORYS to TAO depths and locations

In [ ]:
interp_depths     = adcp_depths          # positive downward (m)
neg_interp_depths = -interp_depths       # TPOSE Z convention

def extract_tpose_vel(ds):
    """Returns (u, v) each (time, location, depth)."""
    ds = ds.isel(time=eval_slice)
    loc_da   = xr.DataArray(tao_lons,          dims='location')
    depth_da = xr.DataArray(neg_interp_depths, dims='depth')

    u = (ds.UVEL
           .sel(YC=0.0, method='nearest')
           .interp(XG=loc_da, Z=depth_da, method='linear')
           .transpose('time', 'location', 'depth')
           .compute()).values

    v = (ds.VVEL
           .sel(YG=0.0, method='nearest')
           .interp(XC=loc_da, Z=depth_da, method='linear')
           .transpose('time', 'location', 'depth')
           .compute()).values

    return u, v

print('TPOSE-noVel state estimate...')
noTAO_u,     noTAO_v     = extract_tpose_vel(ds_tpose_noTAO)
print('TPOSE-noVel forecast...')
noTAO_fct_u, noTAO_fct_v = extract_tpose_vel(ds_tpose_noTAO_forecast)
print('TPOSE-Vel state estimate...')
vel_est_u,   vel_est_v   = extract_tpose_vel(ds_tpose_vel)
print('TPOSE-Vel forecast...')
vel_fct_u,   vel_fct_v   = extract_tpose_vel(ds_tpose_vel_forecast)
print('Done.')

# ── GLORYS extraction ─────────────────────────────────────────────────────────
tao_lons_neg = tao_lons - 360   # [190, 220, 250] → [−170, −140, −110]
loc_da_g   = xr.DataArray(tao_lons_neg, dims='location')
depth_da_g = xr.DataArray(interp_depths, dims='depth')

glorys_u = (glorys_wind.uo
              .sel(latitude=0.0, method='nearest')
              .interp(longitude=loc_da_g, depth=depth_da_g, method='linear')
              .transpose('time', 'location', 'depth')
              .isel(time=eval_slice)
              .compute()).values

glorys_v = (glorys_wind.vo
              .sel(latitude=0.0, method='nearest')
              .interp(longitude=loc_da_g, depth=depth_da_g, method='linear')
              .transpose('time', 'location', 'depth')
              .isel(time=eval_slice)
              .compute()).values

print('GLORYS extracted. Shape:', glorys_u.shape)

# ── HYCOM extraction — skipped if no data for this window ────────────────────
if hycom is not None:
    loc_da_h   = xr.DataArray(tao_lons_neg, dims='location')
    depth_da_h = xr.DataArray(interp_depths, dims='depth')

    hycom_u = (hycom_wind.water_u
                 .sel(lat=0.0, method='nearest')
                 .interp(lon=loc_da_h, depth=depth_da_h, method='linear')
                 .transpose('time', 'location', 'depth')
                 .isel(time=eval_slice)
                 .compute()).values

    hycom_v = (hycom_wind.water_v
                 .sel(lat=0.0, method='nearest')
                 .interp(lon=loc_da_h, depth=depth_da_h, method='linear')
                 .transpose('time', 'location', 'depth')
                 .isel(time=eval_slice)
                 .compute()).values

    print('HYCOM extracted. Shape:', hycom_u.shape)

### Figures: depth–time Hovmöller at each mooring

One figure per mooring longitude (190°E, 220°E, 250°E).  
6 rows × 2 columns: rows = (1) TAO obs, (2) TPOSE-noVel state estimate,  
(3) TPOSE-noVel forecast, (4) TPOSE-Vel state estimate, (5) TPOSE-Vel forecast, (6) GLORYS.  
Left column = U (zonal), right column = V (meridional).

In [ ]:
import cmocean.cm as cmo

row_labels    = ['TAO ADCP', 'TPOSE-noVel State Estimate', 'TPOSE-noVel Forecast',
                 'TPOSE-Vel State Estimate', 'TPOSE-Vel Forecast', 'GLORYS']
forcing_labels = [None, 'optimized forcing', 'reanalysis forcing',
                  'optimized forcing', 'reanalysis forcing', 'optimized forcing']
datasets       = [(tao_u, tao_v), (noTAO_u, noTAO_v), (noTAO_fct_u, noTAO_fct_v),
                  (vel_est_u, vel_est_v), (vel_fct_u, vel_fct_v), (glorys_u, glorys_v)]

if hycom is not None:
    row_labels.append('HYCOM')
    forcing_labels.append('reanalysis forcing')
    datasets.append((hycom_u, hycom_v))

n_rows  = len(datasets)
vlim_u  = 1.2   # m/s
vlim_v  = 0.4   # m/s

days_ax = np.arange(1, n_eval + 1)
depths  = interp_depths   # positive downward (m)

for loc_idx, (lon_e, lon_label) in enumerate(zip(tao_lons, tao_lon_labels)):

    obs_mask_u = ~np.isfinite(tao_u[:, loc_idx, :])
    obs_mask_v = ~np.isfinite(tao_v[:, loc_idx, :])

    fig_height = 3 * n_rows
    fig, axes = plt.subplots(
        n_rows, 2, figsize=(13, fig_height),
        sharex=True, sharey=True,
        gridspec_kw={'hspace': 0.18, 'wspace': 0.15}
    )

    for row, ((u_arr, v_arr), row_label, forcing) in enumerate(zip(datasets, row_labels, forcing_labels)):
        u2d = u_arr[:, loc_idx, :].copy()
        v2d = v_arr[:, loc_idx, :].copy()
        u2d[obs_mask_u] = np.nan
        v2d[obs_mask_v] = np.nan

        for col, (data2d, vlim) in enumerate([(u2d.T, vlim_u), (v2d.T, vlim_v)]):
            ax = axes[row, col]
            pcm = ax.pcolormesh(
                days_ax, depths, np.ma.masked_invalid(data2d),
                cmap=cmo.balance, vmin=-vlim, vmax=vlim, shading='auto'
            )
            fig.colorbar(pcm, ax=ax, fraction=0.03, pad=0.02)
            ax.set_title(row_label, fontsize=13)
            if col == 0:
                ax.set_ylabel('depth [m]')
            if forcing is not None:
                ax.text(0.97, 0.13, forcing, transform=ax.transAxes,
                        ha='right', va='top', fontsize=12,
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                  edgecolor='gray', alpha=0.9))

    axes[0, 0].set_ylim(depths[-1], depths[0])
    for col in range(2):
        axes[-1, col].set_xlabel('Days since start')

    fig.suptitle(
        f'Velocity depth–time  |  {lon_label}  |  Start: {start_date.strftime("%b %d %Y")}',
        fontsize=13, y=0.93
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    for col, col_label in enumerate(['U (m/s)', 'V (m/s)']):
        ax0 = axes[0, col]
        pos = ax0.get_position()
        fig.text(pos.x0 + pos.width / 2, 0.90, col_label,
                 ha='center', va='bottom', fontsize=15, fontweight='bold')

    plt.savefig(
        foldername + f'vel_hovmoller_{int(lon_e):03d}E_{month_str}{day_str}{year_str}.png',
        dpi=150, bbox_inches='tight'
    )
    plt.show()

### Figures: depth–time Hovmöller of model minus TAO observations

Same layout but each panel shows (model − TAO obs). Row 1 is the obs itself (for reference);
rows 2–5 show the bias of each model product relative to the observations.

In [ ]:
import cmocean.cm as cmo

diff_row_labels    = ['TPOSE-noVel State Estimate − Obs', 'TPOSE-noVel Forecast − Obs',
                      'TPOSE-Vel State Estimate − Obs', 'TPOSE-Vel Forecast − Obs', 'GLORYS − Obs']
diff_forcing_labels = ['optimized forcing', 'reanalysis forcing',
                       'optimized forcing', 'reanalysis forcing', 'optimized forcing']
model_datasets      = [(noTAO_u, noTAO_v), (noTAO_fct_u, noTAO_fct_v),
                       (vel_est_u, vel_est_v), (vel_fct_u, vel_fct_v), (glorys_u, glorys_v)]

if hycom is not None:
    diff_row_labels.append('HYCOM − Obs')
    diff_forcing_labels.append('reanalysis forcing')
    model_datasets.append((hycom_u, hycom_v))

n_diff_rows = len(model_datasets)
vlim_diff_u = 0.7   # m/s
vlim_diff_v = 0.4   # m/s

days_ax = np.arange(1, n_eval + 1)
depths  = interp_depths

for loc_idx, (lon_e, lon_label) in enumerate(zip(tao_lons, tao_lon_labels)):

    obs_mask_u = ~np.isfinite(tao_u[:, loc_idx, :])
    obs_mask_v = ~np.isfinite(tao_v[:, loc_idx, :])
    obs_u = tao_u[:, loc_idx, :].copy()
    obs_v = tao_v[:, loc_idx, :].copy()

    fig_height = 3 * n_diff_rows
    fig, axes = plt.subplots(
        n_diff_rows, 2, figsize=(13, fig_height),
        sharex=True, sharey=True,
        gridspec_kw={'hspace': 0.18, 'wspace': 0.15}
    )

    for row, ((u_arr, v_arr), row_label, forcing) in enumerate(
            zip(model_datasets, diff_row_labels, diff_forcing_labels)):
        diff_u = u_arr[:, loc_idx, :] - obs_u
        diff_v = v_arr[:, loc_idx, :] - obs_v
        diff_u[obs_mask_u] = np.nan
        diff_v[obs_mask_v] = np.nan

        for col, (data2d, vlim) in enumerate([(diff_u.T, vlim_diff_u), (diff_v.T, vlim_diff_v)]):
            ax = axes[row, col]
            pcm = ax.pcolormesh(
                days_ax, depths, np.ma.masked_invalid(data2d),
                cmap=cmo.balance, vmin=-vlim, vmax=vlim, shading='auto'
            )
            fig.colorbar(pcm, ax=ax, fraction=0.03, pad=0.02)
            ax.set_title(row_label, fontsize=13)
            if col == 0:
                ax.set_ylabel('depth [m]')
            ax.text(0.97, 0.13, forcing, transform=ax.transAxes,
                    ha='right', va='top', fontsize=12,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                              edgecolor='gray', alpha=0.9))

    axes[0, 0].set_ylim(depths[-1], depths[0])
    for col in range(2):
        axes[-1, col].set_xlabel('Days since start')

    fig.suptitle(
        f'Velocity bias (model − TAO)  |  {lon_label}  |  Start: {start_date.strftime("%b %d %Y")}',
        fontsize=13, y=0.93
    )
    fig.tight_layout(rect=[0, 0, 1, 0.90])

    for col, col_label in enumerate(['U bias (m/s)', 'V bias (m/s)']):
        ax0 = axes[0, col]
        pos = ax0.get_position()
        fig.text(pos.x0 + pos.width / 2, 0.90, col_label,
                 ha='center', va='bottom', fontsize=15, fontweight='bold')

    plt.savefig(
        foldername + f'vel_hovmoller_bias_{int(lon_e):03d}E_{month_str}{day_str}{year_str}.png',
        dpi=150, bbox_inches='tight'
    )
    plt.show()

In [18]:
client.close()
cluster.close()